In [1]:
import sys
import ply.yacc as yacc

class AgroParser(object):
    # --- REGLAS DE NEGOCIO (ANÁLISIS SEMÁNTICO) ---
    # Diccionario con los meses permitidos para cultivar Maíz
    MESES_APTOS_MAIZ = ['Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

    # --- REGLAS GRAMATICALES (PARSER) ---
    
    # Regla inicial: El programa completo es una lista de bloques
    def p_plan(self, p):
        '''plan : bloque_lista'''
        p[0] = p[1] # Transmite la lista de bloques hacia la raíz del parser

    # Regla recursiva: Permite agrupar una lista de múltiples bloques o uno solo
    def p_bloque_lista(self, p):
        '''bloque_lista : bloque bloque_lista

                        | bloque'''
        if len(p) == 3:
            p[0] = [p[1]] + p[2] # Si hay más bloques, los une en una lista secuencial
        else:
            p[0] = [p[1]]        # Si es el último o único bloque, crea la lista inicial

    # Un bloque puede contener una instrucción común o un condicional IF
    def p_bloque(self, p):
        '''bloque : instruction
                  | if_statement'''
        p[0] = p[1] # Pasa el contenido estructurado de la instrucción hacia arriba

    # Estructura sintáctica del IF y el ELSE opcional
    def p_if_statement(self, p):
        '''if_statement : IF LPAREN condition RPAREN LBRACE bloque_lista RBRACE

                        | IF LPAREN condition RPAREN LBRACE bloque_lista RBRACE ELSE LBRACE bloque_lista RBRACE'''
        if len(p) == 8:
            # Si solo tiene IF: Guarda una tupla indicando la condición, el bloque verdadero y una lista vacía para el falso
            p[0] = ('IF', p[3], p[6], [])
        else:
            # Si tiene IF-ELSE: Guarda la condición, el bloque verdadero (p[6]) y el bloque falso (p[10])
            p[0] = ('IF', p[3], p[6], p[10])

    # Estructura de las condiciones (Ej: Temperatura < 3)
    def p_condition(self, p):
        '''condition : VARIABLE OPERATOR value
                     | VARIABLE EQUALS value'''
        p[0] = (p[1], p[2], p[3]) # Estructura la condición como una tupla (Variable, Operador, Valor)

    # Tipos de datos numéricos aceptados en condiciones o litros de riego
    def p_value(self, p):
        '''value : INT
                 | FLOAT'''
        p[0] = p[1] # Retorna el número real (ya transformado por el léxico)

    # --- CATÁLOGO DE INSTRUCCIONES AGRÍCOLAS ---

    # Regla para el comando Sembrar
    def p_instruction_sembrar(self, p):
        '''instruction : SEMBRAR SEMILLA EN MES SEMI'''
        semilla_limpia = p[2].replace('"', '') # Remueve las comillas del token ("Maiz" -> Maiz)
        mes = p[4]
        
        # --- VALIDACIÓN SEMÁNTICA PERSONALIZADA ---
        if semilla_limpia == 'Maiz' and mes not in self.MESES_APTOS_MAIZ:
            print(f"❌ SEMÁNTICO ERROR (Línea {p.lineno(1)}): No se puede sembrar Maíz en {mes}. Meses aptos: {', '.join(self.MESES_APTOS_MAIZ)}.")
        
        p[0] = ('SEMBRAR', semilla_limpia, mes) # Retorna el comando empaquetado en una tupla

    # Regla para el comando Fertilizar
    def p_instruction_fertilizar(self, p):
        '''instruction : FERTILIZAR CON STRING SEMI'''
        p[0] = ('FERTILIZAR', p[3]) # Guarda la acción y el nombre del agroquímico

    # Regla para el comando Fumigar
    def p_instruction_fumigar(self, p):
        '''instruction : FUMIGAR CON STRING POR STRING SEMI'''
        p[0] = ('FUMIGAR', p[3], p[5]) # Guarda (FUMIGAR, Producto, Plaga/Motivo)

    # Regla para el comando de contingencia climática AlertaHelada
    def p_instruction_alerta(self, p):
        '''instruction : ALERTAHELADA APLICAR STRING SEMI'''
        p[0] = ('ALERTAHELADA', p[3]) # Guarda la acción preventiva a ejecutar

    # Regla para el comando Riego
    def p_instruction_riego(self, p):
        '''instruction : RIEGO INT LITROS SEMI'''
        p[0] = ('RIEGO', p[2]) # Guarda la cantidad de litros a regar

    # Función de captura para errores en la estructura del código
    def p_error(self, p):
        if p:
            print(f"SINTÁCTICO: Error de sintaxis en el token '{p.value}' (Línea {p.lineno})")
        else:
            print("SINTÁCTICO: Error inesperado. El código terminó de forma abrupta.")

    # Constructor de la clase sintáctica
    def __init__(self, tokens):
        self.tokens = tokens # Almacena la lista de tokens requerida obligatoriamente por yacc
        
        # 💡 PARCHE DE COMPATIBILIDAD PYTHON 3.14 PARA EL PARSER:
        # Inyecta un archivo virtual en el entorno interactivo para que yacc no falle al inicializarse
        main_module = sys.modules['__main__']
        if not hasattr(main_module, '__file__'):
            main_module.__file__ = 'notebook_fake_file.py'
            
        # Construye el motor sintáctico yacc vinculándolo a esta instancia clase sin generar archivos caché (.pyc)
        self.parser = yacc.yacc(module=self, write_tables=False, debug=False)
